# ⚡ OpenMythos em 5 Minutos
Versão turbo para testar a ideia sem enrolação.

In [ ]:
!pip install open-mythos -q
import torch
from open_mythos import MythosConfig, OpenMythos

cfg = MythosConfig(vocab_size=500, dim=64, n_heads=2, n_kv_heads=1,
                   max_seq_len=32, max_loop_iters=2, prelude_layers=1,
                   coda_layers=1, attn_type="gqa", n_experts=2,
                   n_shared_experts=1, n_experts_per_tok=1, expert_dim=32, lora_rank=2)

model = OpenMythos(cfg).to("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ {sum(p.numel() for p in model.parameters()):,} parâmetros")

In [ ]:
frases = ["o ceu e azul", "o sol brilha", "a lua e cheia",
          "o gato mia", "o cachorro late", "a flor e bonita"] * 3

vocab = {"<PAD>":0,"<BOS>":2,"<EOS>":3}; [vocab.setdefault(w, len(vocab)) for f in frases for w in f.split()]
inv = {v:k for k,v in vocab.items()}

def enc(t): return [vocab.get(w,1) for w in t.split()]
X = torch.zeros(len(frases), 32, dtype=torch.long)
Y = torch.zeros(len(frases), 32, dtype=torch.long)
for i,f in enumerate(frases):
    t = [2] + enc(f) + [3]; pad = 32-len(t)
    X[i] = torch.tensor(t[:31] + [0]*pad)
    Y[i] = torch.tensor(t[1:32] + [0]*pad)
print(f"📚 {len(frases)} frases, vocabulário: {len(vocab)} tokens")

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
for i in range(30):
    opt.zero_grad()
    loss = torch.nn.functional.cross_entropy(model(X, n_loops=2).reshape(-1,cfg.vocab_size), Y.reshape(-1))
    loss.backward(); opt.step()
    if (i+1)%10==0: print(f"Step {i+1}: loss {loss.item():.4f}")
print(f"✅ Treino concluído. Loss final: {loss.item():.4f}")

In [ ]:
inp = torch.tensor([[2]+enc("o ceu")], device=next(model.parameters()).device)
out = model.generate(inp, max_new_tokens=8, n_loops=2, temperature=0.8, top_k=10)
print("🔮", " ".join(inv.get(int(t),"?") for t in out[0] if int(t) not in (0,2,3)))